In [1]:
import pandas as pd
import json

# -------------------------------
# 1️⃣ LOAD JSON FILE
# -------------------------------
with open(r'C:\ProgramData\MySQL\MySQL Server 8.0\Uploads\agents_20k.json', 'r') as f:
    data = json.load(f)

# Convert single JSON object to list
if isinstance(data, dict):
    data = [data]

# -------------------------------
# 2️⃣ FLATTEN JSON
# -------------------------------
df = pd.json_normalize(data)

print("Original Data Preview:")
print(df.head())

# -------------------------------
# 3️⃣ STANDARDIZE COLUMN NAMES
# -------------------------------
df.columns = (
    df.columns.str.lower()
              .str.strip()
              .str.replace(" ", "_")
)

# -------------------------------
# 4️⃣ CONVERT NUMERIC COLUMNS
# -------------------------------
numeric_cols = [
    'bhk',
    'size_in_sqft',
    'price_in_lakhs',
    'price_per_sqft',
    'year_built',
    'floor_no',
    'total_floors',
    'age_of_property',
    'nearby_schools',
    'nearby_hospitals',
    'public_transport_accessibility'
]

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

# -------------------------------
# 5️⃣ BOOLEAN COLUMN CLEANING
# -------------------------------
bool_cols = ['parking_space', 'security']

for col in bool_cols:
    if col in df.columns:
        df[col] = df[col].astype(str).str.lower().map({
            'yes': True,
            'true': True,
            '1': True,
            'no': False,
            'false': False,
            '0': False
        })

# -------------------------------
# 6️⃣ TEXT CLEANING
# -------------------------------
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].astype(str).str.strip().str.lower()

# -------------------------------
# 7️⃣ HANDLE MISSING VALUES
# -------------------------------
numeric_fill_cols = [
    'bhk',
    'size_in_sqft',
    'price_in_lakhs',
    'price_per_sqft'
]

for col in numeric_fill_cols:
    if col in df.columns:
        df[col].fillna(0, inplace=True)

df.fillna("unknown", inplace=True)

# -------------------------------
# 8️⃣ REMOVE DUPLICATES
# -------------------------------
if 'id' in df.columns:
    df.drop_duplicates(subset=['id'], inplace=True)

# -------------------------------
# 9️⃣ REMOVE INVALID VALUES
# -------------------------------
if 'price_in_lakhs' in df.columns:
    df = df[df['price_in_lakhs'] >= 0]

if 'size_in_sqft' in df.columns:
    df = df[df['size_in_sqft'] >= 0]

# -------------------------------
# 🔟 SAVE CLEANED DATA
# -------------------------------
df.to_csv("cleaned_agents.csv", index=False)

print("\n✅ Data Preparation Completed Successfully!")
print(df.info())
print(df.head())

Original Data Preview:
  Agent_ID         Name            Phone                 Email
0    A0001  Agent A0001  +1-534-665-8373  a0001@realestate.com
1    A0002  Agent A0002  +1-493-463-4698  a0002@realestate.com
2    A0003  Agent A0003  +1-290-534-1121  a0003@realestate.com
3    A0004  Agent A0004  +1-691-610-4878  a0004@realestate.com
4    A0005  Agent A0005  +1-829-613-5411  a0005@realestate.com

✅ Data Preparation Completed Successfully!
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   agent_id  20000 non-null  object
 1   name      20000 non-null  object
 2   phone     20000 non-null  object
 3   email     20000 non-null  object
dtypes: object(4)
memory usage: 625.1+ KB
None
  agent_id         name            phone                 email
0    a0001  agent a0001  +1-534-665-8373  a0001@realestate.com
1    a0002  agent a0002  +1-493-463-4698  a000

In [2]:
import pandas as pd
import json

# -------------------------------
# 1️⃣ LOAD JSON FILE
# -------------------------------
with open(r'C:\ProgramData\MySQL\MySQL Server 8.0\Uploads\buyers_20k.json', 'r') as f:
    data = json.load(f)

# Convert single object to list
if isinstance(data, dict):
    data = [data]

# -------------------------------
# 2️⃣ FLATTEN JSON
# -------------------------------
df = pd.json_normalize(data)

print("Original Data Preview:")
print(df.head())

# -------------------------------
# 3️⃣ STANDARDIZE COLUMN NAMES
# -------------------------------
df.columns = (
    df.columns.str.lower()
              .str.strip()
              .str.replace(" ", "_")
)

# -------------------------------
# 4️⃣ CONVERT NUMERIC COLUMNS
# -------------------------------
numeric_cols = [
    'buyer_id',
    'loan_amount'
]

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

# -------------------------------
# 5️⃣ BOOLEAN COLUMN CLEANING
# -------------------------------
bool_cols = ['loan_taken']

for col in bool_cols:
    if col in df.columns:
        df[col] = df[col].astype(str).str.lower().map({
            'yes': True,
            'true': True,
            '1': True,
            'no': False,
            'false': False,
            '0': False
        })

# -------------------------------
# 6️⃣ TEXT CLEANING
# -------------------------------
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].astype(str).str.strip().str.lower()

# -------------------------------
# 7️⃣ HANDLE MISSING VALUES
# -------------------------------
for col in numeric_cols:
    if col in df.columns:
        df[col].fillna(0, inplace=True)

if 'loan_provider' in df.columns:
    df['loan_provider'].fillna('not_applicable', inplace=True)

df.fillna("unknown", inplace=True)

# -------------------------------
# 8️⃣ REMOVE DUPLICATES
# -------------------------------
if 'sale_id' in df.columns:
    df.drop_duplicates(subset=['sale_id'], inplace=True)

# -------------------------------
# 9️⃣ REMOVE INVALID VALUES
# -------------------------------
if 'loan_amount' in df.columns:
    df = df[df['loan_amount'] >= 0]

# Ensure no loan amount if loan not taken
if 'loan_taken' in df.columns and 'loan_amount' in df.columns:
    df.loc[df['loan_taken'] == False, 'loan_amount'] = 0

# -------------------------------
# 🔟 SAVE CLEANED DATA
# -------------------------------
df.to_csv("cleaned_buyers.csv", index=False)

print("\n✅ Data Preparation Completed Successfully!")
print(df.info())
print(df.head())

Original Data Preview:
   buyer_id sale_id buyer_type   payment_mode  loan_taken loan_provider  \
0         1  L01179   End User           Cash       False          None   
1         2  L00866   Investor         Cheque       False          None   
2         3  L00102   Investor         Cheque        True          Axis   
3         4  L00440   Investor  Bank Transfer       False          None   
4         5  L00059   Investor            UPI        True          HDFC   

   loan_amount  
0            0  
1            0  
2      2317757  
3            0  
4      4191221  

✅ Data Preparation Completed Successfully!
<class 'pandas.core.frame.DataFrame'>
Index: 720 entries, 0 to 719
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   buyer_id       720 non-null    int64 
 1   sale_id        720 non-null    object
 2   buyer_type     720 non-null    object
 3   payment_mode   720 non-null    object
 4   loan_taken     720

C:\Users\GLOBAL\AppData\Local\Temp\ipykernel_18780\2664467193.py:70: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(0, inplace=True)
C:\Users\GLOBAL\AppData\Local\Temp\ipykernel_18780\2664467193.py:73: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when do

In [3]:
import pandas as pd
import json

# -------------------------------
# 1️⃣ LOAD JSON FILE
# -------------------------------
with open(r'C:\ProgramData\MySQL\MySQL Server 8.0\Uploads\property_attributes_20k.json', 'r') as f:
    data = json.load(f)

# Convert single object to list
if isinstance(data, dict):
    data = [data]

# -------------------------------
# 2️⃣ FLATTEN JSON
# -------------------------------
df = pd.json_normalize(data)

print("Original Data Preview:")
print(df.head())

# -------------------------------
# 3️⃣ STANDARDIZE COLUMN NAMES
# -------------------------------
df.columns = (
    df.columns.str.lower()
              .str.strip()
              .str.replace(" ", "_")
)

# -------------------------------
# 4️⃣ CONVERT NUMERIC COLUMNS
# -------------------------------
numeric_cols = [
    'attribute_id',
    'bedrooms',
    'bathrooms',
    'floor_number',
    'total_floors',
    'year_built',
    'tenant_count',
    'metro_distance_km'
]

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

# -------------------------------
# 5️⃣ BOOLEAN COLUMN CLEANING
# -------------------------------
bool_cols = [
    'is_rented',
    'parking_available',
    'power_backup'
]

for col in bool_cols:
    if col in df.columns:
        df[col] = df[col].astype(str).str.lower().map({
            'yes': True,
            'true': True,
            '1': True,
            'no': False,
            'false': False,
            '0': False
        })

# -------------------------------
# 6️⃣ TEXT CLEANING
# -------------------------------
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].astype(str).str.strip().str.lower()

# -------------------------------
# 7️⃣ HANDLE MISSING VALUES
# -------------------------------
for col in numeric_cols:
    if col in df.columns:
        df[col].fillna(0, inplace=True)

if 'furnishing_status' in df.columns:
    df['furnishing_status'].fillna('unknown', inplace=True)

df.fillna("unknown", inplace=True)

# -------------------------------
# 8️⃣ REMOVE DUPLICATES
# -------------------------------
if 'attribute_id' in df.columns:
    df.drop_duplicates(subset=['attribute_id'], inplace=True)

# -------------------------------
# 9️⃣ LOGICAL VALIDATION
# -------------------------------

# No negative values
for col in numeric_cols:
    if col in df.columns:
        df = df[df[col] >= 0]

# Floor number cannot exceed total floors
if 'floor_number' in df.columns and 'total_floors' in df.columns:
    df = df[df['floor_number'] <= df['total_floors']]

# Tenant count should be zero if not rented
if 'is_rented' in df.columns and 'tenant_count' in df.columns:
    df.loc[df['is_rented'] == False, 'tenant_count'] = 0

# Year built sanity check
if 'year_built' in df.columns:
    df = df[(df['year_built'] >= 1800) & (df['year_built'] <= 2026)]

# -------------------------------
# 🔟 SAVE CLEANED DATA
# -------------------------------
df.to_csv("cleaned_property_attributes.csv", index=False)

print("\n✅ Data Preparation Completed Successfully!")
print(df.info())
print(df.head())

Original Data Preview:
   attribute_id listing_id  bedrooms  bathrooms  floor_number  total_floors  \
0             1     L00001         5          3             9             9   
1             2     L00002         2          2            19            29   
2             3     L00003         2          3             8            26   
3             4     L00004         3          3            25            10   
4             5     L00005         2          2            15            16   

   year_built  is_rented  tenant_count furnishing_status  metro_distance_km  \
0        2001       True             4         Furnished               7.26   
1        2020      False             0       Unfurnished               4.84   
2        2007       True             4    Semi-Furnished               4.92   
3        2003      False             3         Furnished               1.20   
4        2008       True             1    Semi-Furnished               7.90   

   parking_available  power

C:\Users\GLOBAL\AppData\Local\Temp\ipykernel_18780\1732371503.py:80: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(0, inplace=True)
C:\Users\GLOBAL\AppData\Local\Temp\ipykernel_18780\1732371503.py:83: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when do


✅ Data Preparation Completed Successfully!
<class 'pandas.core.frame.DataFrame'>
Index: 15495 entries, 0 to 19999
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   attribute_id       15495 non-null  int64  
 1   listing_id         15495 non-null  object 
 2   bedrooms           15495 non-null  int64  
 3   bathrooms          15495 non-null  int64  
 4   floor_number       15495 non-null  int64  
 5   total_floors       15495 non-null  int64  
 6   year_built         15495 non-null  int64  
 7   is_rented          15495 non-null  bool   
 8   tenant_count       15495 non-null  int64  
 9   furnishing_status  15495 non-null  object 
 10  metro_distance_km  15495 non-null  float64
 11  parking_available  15495 non-null  bool   
 12  power_backup       15495 non-null  bool   
dtypes: bool(3), float64(1), int64(7), object(2)
memory usage: 1.3+ MB
None
   attribute_id listing_id  bedrooms  bathrooms  floor_n

In [4]:
import pandas as pd
import json

# -------------------------------
# 1️⃣ LOAD JSON FILE
# -------------------------------
with open(r'C:\ProgramData\MySQL\MySQL Server 8.0\Uploads\listings_20k.json', 'r') as f:
    data = json.load(f)

# If JSON is single object → convert to list
if isinstance(data, dict):
    data = [data]

# -------------------------------
# 2️⃣ FLATTEN JSON
# -------------------------------
df = pd.json_normalize(data)

print("Original Data:")
print(df.head())

# -------------------------------
# 3️⃣ STANDARDIZE COLUMN NAMES
# -------------------------------
df.columns = df.columns.str.lower().str.strip().str.replace(" ", "_")

# -------------------------------
# 4️⃣ HANDLE DATA TYPES
# -------------------------------

# Convert numeric columns
numeric_cols = ['price', 'sqft', 'area', 'latitude', 'longitude']
for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

# Convert date columns
if 'date_listed' in df.columns:
    df['date_listed'] = pd.to_datetime(df['date_listed'], errors='coerce')

# Convert boolean columns
bool_cols = ['is_rented', 'parking_available', 'power_backup']
for col in bool_cols:
    if col in df.columns:
        df[col] = df[col].astype(bool)

# -------------------------------
# 5️⃣ CLEAN TEXT DATA
# -------------------------------
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].str.strip().str.lower()

# -------------------------------
# 6️⃣ HANDLE MISSING VALUES
# -------------------------------
df.fillna({
    'price': 0,
    'sqft': 0,
    'area': 0
}, inplace=True)

df.fillna("unknown", inplace=True)

# -------------------------------
# 7️⃣ REMOVE DUPLICATES
# -------------------------------
if 'listing_id' in df.columns:
    df.drop_duplicates(subset=['listing_id'], inplace=True)

# -------------------------------
# 8️⃣ ENSURE CONSISTENCY
# -------------------------------

# Fix negative or invalid values
if 'price' in df.columns:
    df = df[df['price'] >= 0]

if 'sqft' in df.columns:
    df = df[df['sqft'] >= 0]

# -------------------------------
# 9️⃣ SAVE CLEAN DATA
# -------------------------------
df.to_csv("cleaned_listings.csv", index=False)

print("\n✅ Data Preparation Completed Successfully!")
print(df.info())

Original Data:
  Listing_ID         City Property_Type         Price         Sqft  \
0     L00001     New York     Apartment  1.655144e+06  2753.009121   
1     L00002  Los Angeles     Apartment  1.519141e+06  4966.988193   
2     L00003      Houston     Apartment  1.624890e+05  1267.003959   
3     L00004      Phoenix     Apartment  1.277016e+06  2128.014429   
4     L00005      Phoenix     Townhouse  5.622970e+05  4178.997421   

  Date_Listed Agent_ID   Latitude   Longitude  
0  2023-05-06    A0015  33.965208  -69.861589  
1  2023-02-14    A0038  42.547892  -90.277860  
2  2023-04-22    A0015  28.732327 -115.952982  
3  2024-01-02    A0042  26.403938  -74.771490  
4  2023-10-29    A0018  39.425252  -83.917878  

✅ Data Preparation Completed Successfully!
<class 'pandas.core.frame.DataFrame'>
Index: 1200 entries, 0 to 1199
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   listing_id     1200 non-

In [5]:
import pandas as pd
import json

# -------------------------------
# 1️⃣ LOAD DATA
# -------------------------------
# The original code tried to load a CSV file as JSON, causing a JSONDecodeError.
# We will now use pandas to correctly read the CSV file.
# with open('/content/drive/MyDrive/Guvi/Dataset/sales_20k.csv', 'r') as f:
#     data = json.load(f)

# Convert single object to list (This part is for JSON. For CSV, directly load into DataFrame)
# if isinstance(data, dict):
#     data = [data]

df = pd.read_csv(r'C:\ProgramData\MySQL\MySQL Server 8.0\Uploads\sales_20k.csv')

# -------------------------------
# 2️⃣ FLATTEN JSON (Not applicable for CSV, but keeping structure for consistency if JSON was used)
# -------------------------------
# If the CSV had nested JSON strings in columns, we would process them here.
# For a flat CSV, json_normalize is not needed directly at this step for the whole df.
# df = pd.json_normalize(data)

print("Original Data Preview:")
print(df.head())

# -------------------------------
# 3️⃣ STANDARDIZE COLUMN NAMES
# -------------------------------
df.columns = (
    df.columns.str.lower()
              .str.strip()
              .str.replace(" ", "_")
)

# -------------------------------
# 4️⃣ CONVERT NUMERIC COLUMNS
# -------------------------------
numeric_cols = [
    'sale_price',
    'days_on_market'
]

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

# -------------------------------
# 5️⃣ STANDARDIZE DATE FORMAT
# -------------------------------
if 'date_sold' in df.columns:
    df['date_sold'] = pd.to_datetime(
        df['date_sold'],
        format='%d-%m-%Y',
        errors='coerce'
    )

# Optional: Convert to consistent format
df['date_sold'] = df['date_sold'].dt.strftime('%Y-%m-%d')

# -------------------------------
# 6️⃣ TEXT CLEANING
# -------------------------------
for col in df.select_dtypes(include='object').columns:
    if col != 'date_sold':
        df[col] = df[col].astype(str).str.strip().str.lower()

# -------------------------------
# 7️⃣ HANDLE MISSING VALUES
# -------------------------------
for col in numeric_cols:
    if col in df.columns:
        df[col] = df[col].fillna(0) # Removed inplace=True and assigned back

df = df.fillna("unknown") # Removed inplace=True and assigned back

# -------------------------------
# 8️⃣ REMOVE DUPLICATES
# -------------------------------
if 'listing_id' in df.columns:
    df.drop_duplicates(subset=['listing_id'], inplace=True)

# -------------------------------
# 9️⃣ REMOVE INVALID VALUES
# -------------------------------
if 'sale_price' in df.columns:
    df = df[df['sale_price'] > 0]

if 'days_on_market' in df.columns:
    df = df[df['days_on_market'] >= 0]

# -------------------------------
# 🔟 ROUND FLOAT VALUES
# -------------------------------
if 'sale_price' in df.columns:
    df['sale_price'] = df['sale_price'].round(2)

if 'days_on_market' in df.columns:
    df['days_on_market'] = df['days_on_market'].round(2)

# -------------------------------
# 1️⃣1️⃣ SAVE CLEANED DATA
# -------------------------------
df.to_csv("cleaned_sales_data.csv", index=False)

print("\n✅ Data Preparation Completed Successfully!")
print(df.info())
print(df.head())

Original Data Preview:
  Listing_ID    Sale_Price   Date_Sold  Days_on_Market
0     L01179  9.255800e+05  2023-07-07       65.005560
1     L00866  1.054160e+05  2023-06-14       38.004620
2     L00102  1.825184e+06  2023-09-09       22.992622
3     L00440  1.932085e+06  2023-10-29       72.012274
4     L00059  7.765860e+05  2023-05-01      116.000152

✅ Data Preparation Completed Successfully!
<class 'pandas.core.frame.DataFrame'>
Index: 720 entries, 0 to 719
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   listing_id      720 non-null    object 
 1   sale_price      720 non-null    float64
 2   date_sold       720 non-null    object 
 3   days_on_market  720 non-null    float64
dtypes: float64(2), object(2)
memory usage: 28.1+ KB
None
  listing_id  sale_price date_sold  days_on_market
0     l01179   925580.00   unknown           65.01
1     l00866   105416.00   unknown           38.00
2     l00102  1825184.0

In [6]:
import pandas as pd
agents = pd.read_json( r"C:\ProgramData\MySQL\MySQL Server 8.0\Uploads\agents_20k.json")
listings = pd.read_json(r"C:\ProgramData\MySQL\MySQL Server 8.0\Uploads\listings_20k.json")
sales = pd.read_csv(r"C:\ProgramData\MySQL\MySQL Server 8.0\Uploads\sales_20k.csv")
buyers = pd.read_json(r"C:\ProgramData\MySQL\MySQL Server 8.0\Uploads\buyers_20k.json")
property = pd.read_json(r"C:\ProgramData\MySQL\MySQL Server 8.0\Uploads\property_attributes_20k.json")

In [7]:
# ============================================
# CONNECT MYSQL IN JUPYTER NOTEBOOK (.ipynb)
# ============================================

# Step 1️⃣ Install MySQL Connector
# Run this only once

!pip install mysql-connector-python


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
# Step 2️⃣ Import Libraries

import mysql.connector
import pandas as pd

In [9]:
import mysql.connector

conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="Royal@26"
)

cursor = conn.cursor()

print("✅ MySQL Connected")

✅ MySQL Connected


In [10]:
# ==========================================================
# CREATE DATABASE
# ==========================================================

cursor.execute("DROP DATABASE IF EXISTS real_estate_db")

cursor.execute("CREATE DATABASE real_estate_db")

cursor.execute("USE real_estate_db")

print("✅ Database Created Successfully")

✅ Database Created Successfully


In [11]:
cursor.execute("SHOW DATABASES")

for db in cursor.fetchall():
    print(db)

('information_schema',)
('market_stars_schema',)
('mysql',)
('performance_schema',)
('real_estate_db',)
('real_estates_db',)
('sql_lite',)
('sys',)


In [12]:
# ==========================================================
# CREATE AGENTS TABLE
# ==========================================================

cursor.execute("""
CREATE TABLE agents (
    Agent_ID VARCHAR(10) PRIMARY KEY,
    Name VARCHAR(100),
    Phone VARCHAR(20),
    Email VARCHAR(100)
)
""")

print("✅ agents table created")

✅ agents table created


In [13]:
# ==========================================================
# CREATE LISTINGS TABLE
# ==========================================================

cursor.execute("""
CREATE TABLE listings (
    Listing_ID VARCHAR(10) PRIMARY KEY,
    City VARCHAR(50),
    Property_Type VARCHAR(50),
    Price DECIMAL(15,2),
    Sqft DECIMAL(10,2),
    Date_Listed DATE,
    Agent_ID VARCHAR(10),
    Latitude DECIMAL(10,8),
    Longitude DECIMAL(11,8),

    FOREIGN KEY (Agent_ID)
    REFERENCES agents(Agent_ID)
)
""")

print("✅ listings table created")

✅ listings table created


In [14]:
# ==========================================================
# CREATE PROPERTY ATTRIBUTES TABLE
# ==========================================================

cursor.execute("""
CREATE TABLE property_attributes (
    attribute_id INT PRIMARY KEY,
    listing_id VARCHAR(10),

    bedrooms INT,
    bathrooms INT,
    floor_number INT,
    total_floors INT,
    year_built YEAR,

    is_rented BOOLEAN,
    tenant_count INT,

    furnishing_status VARCHAR(50),

    metro_distance_km DECIMAL(5,2),

    parking_available BOOLEAN,
    power_backup BOOLEAN,

    FOREIGN KEY (listing_id)
    REFERENCES listings(Listing_ID)
)
""")

print("✅ property_attributes table created")

✅ property_attributes table created


In [15]:
# ==========================================================
# CREATE SALES TABLE
# ==========================================================

cursor.execute("""
CREATE TABLE sales_data (
    Listing_ID VARCHAR(10) PRIMARY KEY,

    Sale_Price DECIMAL(15,2),
    Date_Sold DATE,
    Days_on_Market DECIMAL(6,2),

    FOREIGN KEY (Listing_ID)
    REFERENCES listings(Listing_ID)
)
""")

print("✅ sales_data table created")

✅ sales_data table created


In [16]:
# ==========================================================
# CREATE BUYERS TABLE
# ==========================================================

cursor.execute("""
CREATE TABLE buyers (
    buyer_id INT,
    sale_id VARCHAR(10),

    buyer_type VARCHAR(50),
    payment_mode VARCHAR(50),

    loan_taken BOOLEAN,
    loan_provider VARCHAR(100),
    loan_amount DECIMAL(15,2),

    PRIMARY KEY (buyer_id, sale_id),

    FOREIGN KEY (sale_id)
    REFERENCES sales_data(Listing_ID)
)
""")

print("✅ buyers table created")

✅ buyers table created


In [17]:
# ==========================================================
# SHOW ALL TABLES
# ==========================================================

cursor.execute("SHOW TABLES")

tables = cursor.fetchall()

print("\n📋 TABLES & VIEWS IN DATABASE:\n")

for table in tables:
    print(table[0])


📋 TABLES & VIEWS IN DATABASE:

agents
buyers
listings
property_attributes
sales_data


In [18]:
# ==========================================================
# LOAD CLEANED LISTINGS CSV
# ==========================================================
import pandas as pd
agents_df = pd.read_csv("B:\Job\GUVI\Project\Realestate\cleaned_agents.csv")
listings_df = pd.read_csv("B:\Job\GUVI\Project\Realestate\cleaned_listings.csv")
sales_data_df = pd.read_csv("B:\Job\GUVI\Project\Realestate\cleaned_sales_data.csv")
buyers_df = pd.read_csv("B:\Job\GUVI\Project\Realestate\cleaned_buyers.csv")
property_attributes_df = pd.read_csv("B:\Job\GUVI\Project\Realestate\cleaned_property_attributes.csv")


In [19]:
# ==========================================================
# STANDARDIZE COLUMN NAMES
# ==========================================================

agents_df.columns = (
    agents_df.columns
    .str.lower()
    .str.strip()
    .str.replace(" ", "_")
)

listings_df.columns = (
    listings_df.columns
    .str.lower()
    .str.strip()
    .str.replace(" ", "_")
)

sales_data_df.columns = (
    sales_data_df.columns
    .str.lower()
    .str.strip()
    .str.replace(" ", "_")
)

buyers_df.columns = (
    buyers_df.columns
    .str.lower()
    .str.strip()
    .str.replace(" ", "_")
)

property_attributes_df.columns = (
    property_attributes_df.columns
    .str.lower()
    .str.strip()
    .str.replace(" ", "_")
)
print(agents_df.columns)
print(listings_df.columns)
print(sales_data_df.columns)
print(buyers_df.columns)
print(property_attributes_df.columns)

Index(['agent_id', 'name', 'phone', 'email'], dtype='object')
Index(['listing_id', 'city', 'property_type', 'price', 'sqft', 'date_listed',
       'agent_id', 'latitude', 'longitude'],
      dtype='object')
Index(['listing_id', 'sale_price', 'date_sold', 'days_on_market'], dtype='object')
Index(['buyer_id', 'sale_id', 'buyer_type', 'payment_mode', 'loan_taken',
       'loan_provider', 'loan_amount'],
      dtype='object')
Index(['attribute_id', 'listing_id', 'bedrooms', 'bathrooms', 'floor_number',
       'total_floors', 'year_built', 'is_rented', 'tenant_count',
       'furnishing_status', 'metro_distance_km', 'parking_available',
       'power_backup'],
      dtype='object')


In [20]:
# ==========================================================
# CONVERT DATE COLUMN
# ==========================================================

listings_df['date_listed'] = pd.to_datetime(
    listings_df['date_listed'],
    errors='coerce'
).dt.date
sales_data_df['days_on_market'] = pd.to_numeric(
    sales_data_df['days_on_market'],
    errors='coerce'
)

sales_data_df['date_sold'] = pd.to_datetime(
    sales_data_df['date_sold'],
    errors='coerce'
).dt.date

In [21]:
cursor.execute("DELETE FROM agents")

conn.commit()

print("✅ Old records deleted")

✅ Old records deleted


In [22]:

agents_df.drop_duplicates(
    subset=['agent_id'],
    inplace=True
)

In [23]:
for i, row in agents_df.iterrows():
    cursor.execute("""
    insert ignore into agents (agent_id, name, phone, email)
    values (%s,%s,%s,%s)            
                 """,
    (row['agent_id'], row['name'], row['phone'], row['email']))   
conn.commit()

In [24]:
for i, row in listings_df.iterrows():
    cursor.execute("""
    insert into listings (listing_id,city,property_type,price,sqft,date_listed,agent_id,latitude,longitude)
    values (%s,%s,%s,%s,%s,%s,%s,%s,%s)            
                 """,
    (row['listing_id'], row['city'], row['property_type'], row['price'], row['sqft'], row['date_listed'],
      row['agent_id'], row['latitude'], row['longitude']) )      
conn.commit()

In [25]:
for i, row in sales_data_df.iterrows():
    cursor.execute("""
    insert into sales_data (listing_id, sale_price, date_sold, days_on_market)
    values (%s,%s,%s,%s)            
                 """,
    (row['listing_id'], row['sale_price'], row['date_sold'], row['days_on_market']) )      
conn.commit()

In [26]:
for i, row in buyers_df.iterrows():
    cursor.execute("""
    insert into buyers (buyer_id, sale_id, buyer_type, payment_mode, loan_taken,
       loan_provider, loan_amount)
    values (%s,%s,%s,%s,%s,%s,%s)            
                 """,
    (row['buyer_id'], row['sale_id'], row['buyer_type'], row['payment_mode'], row['loan_taken'], row['loan_provider'], row['loan_amount']) )      
conn.commit()

In [27]:
for i, row in property_attributes_df.iterrows():
    cursor.execute("""
    insert into property_attributes (attribute_id, listing_id, bedrooms, bathrooms, floor_number,
       total_floors, year_built, is_rented, tenant_count,furnishing_status, metro_distance_km, parking_available,
       power_backup)
    values (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)            
                 """,
    (row['attribute_id'], row['listing_id'], row['bedrooms'], row['bathrooms'], row['floor_number'], row['total_floors'],
      row['year_built'], row['is_rented'], row['tenant_count'], row['furnishing_status'], row['metro_distance_km'], row['parking_available'], row['power_backup']) )      
conn.commit()

In [28]:
# ==========================================================
# 1️⃣ Average Listing Price by City
# ==========================================================

query = """
SELECT 
    City,
    AVG(Price) AS Avg_Listing_Price
FROM listings
GROUP BY City
ORDER BY Avg_Listing_Price DESC
"""

df1 = pd.read_sql(query, conn)

print(df1)

          City  Avg_Listing_Price
0  los angeles       1.093039e+06
1      houston       1.082539e+06
2      phoenix       1.071594e+06
3      chicago       1.065800e+06
4     new york       1.036485e+06


C:\Users\GLOBAL\AppData\Local\Temp\ipykernel_18780\1432384503.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df1 = pd.read_sql(query, conn)


In [29]:
# ============================================
# 📊 PROPERTY & PRICING ANALYSIS
# ============================================

import pandas as pd

# 1️⃣ Average listing price by city
query1 = """
SELECT 
    City,
    AVG(Price) AS Avg_Listing_Price
FROM listings
GROUP BY City
ORDER BY Avg_Listing_Price DESC
"""

df1 = pd.read_sql(query1, conn)
print("\n1️⃣ Average Listing Price by City")
print(df1)



# 2️⃣ Average price per square foot by property type
query2 = """
SELECT 
    Property_Type,
    AVG(Price / Sqft) AS Avg_Price_Per_Sqft
FROM listings
GROUP BY Property_Type
"""

df2 = pd.read_sql(query2, conn)
print("\n2️⃣ Average Price per Sqft by Property Type")
print(df2)



# 3️⃣ Furnishing status impact on property prices
query3 = """
SELECT 
    furnishing_status,
    AVG(l.Price) AS Avg_Price
FROM property_attributes p
JOIN listings l
ON p.listing_id = l.Listing_ID
GROUP BY furnishing_status
"""

df3 = pd.read_sql(query3, conn)
print("\n3️⃣ Furnishing Status Impact")
print(df3)



# 4️⃣ Metro distance vs property prices
query4 = """
SELECT 
    metro_distance_km,
    AVG(l.Price) AS Avg_Price
FROM property_attributes p
JOIN listings l
ON p.listing_id = l.Listing_ID
GROUP BY metro_distance_km
ORDER BY metro_distance_km
"""

df4 = pd.read_sql(query4, conn)
print("\n4️⃣ Metro Distance vs Price")
print(df4)



# 5️⃣ Rented vs non-rented pricing
query5 = """
SELECT 
    is_rented,
    AVG(l.Price) AS Avg_Price
FROM property_attributes p
JOIN listings l
ON p.listing_id = l.Listing_ID
GROUP BY is_rented
"""

df5 = pd.read_sql(query5, conn)
print("\n5️⃣ Rented vs Non-Rented Prices")
print(df5)



# 6️⃣ Bedrooms & bathrooms effect on pricing
query6 = """
SELECT 
    bedrooms,
    bathrooms,
    AVG(l.Price) AS Avg_Price
FROM property_attributes p
JOIN listings l
ON p.listing_id = l.Listing_ID
GROUP BY bedrooms, bathrooms
ORDER BY Avg_Price DESC
"""

df6 = pd.read_sql(query6, conn)
print("\n6️⃣ Bedrooms & Bathrooms Effect")
print(df6)



# 7️⃣ Parking & power backup impact
query7 = """
SELECT 
    parking_available,
    power_backup,
    AVG(l.Price) AS Avg_Price
FROM property_attributes p
JOIN listings l
ON p.listing_id = l.Listing_ID
GROUP BY parking_available, power_backup
"""

df7 = pd.read_sql(query7, conn)
print("\n7️⃣ Parking & Power Backup Impact")
print(df7)



# 8️⃣ Year built influence on price
query8 = """
SELECT 
    year_built,
    AVG(l.Price) AS Avg_Price
FROM property_attributes p
JOIN listings l
ON p.listing_id = l.Listing_ID
GROUP BY year_built
ORDER BY year_built
"""

df8 = pd.read_sql(query8, conn)
print("\n8️⃣ Year Built Influence")
print(df8)



# 9️⃣ Cities with highest average property prices
query9 = """
SELECT 
    City,
    AVG(Price) AS Avg_Price
FROM listings
GROUP BY City
ORDER BY Avg_Price DESC
LIMIT 10
"""

df9 = pd.read_sql(query9, conn)
print("\n9️⃣ Highest Average Property Prices")
print(df9)



# 🔟 Properties across price buckets
query10 = """
SELECT 
    CASE
        WHEN Price < 100000 THEN 'Below 100K'
        WHEN Price BETWEEN 100000 AND 500000 THEN '100K-500K'
        WHEN Price BETWEEN 500001 AND 1000000 THEN '500K-1M'
        ELSE 'Above 1M'
    END AS Price_Bucket,
    
    COUNT(*) AS Property_Count

FROM listings

GROUP BY Price_Bucket
"""

df10 = pd.read_sql(query10, conn)
print("\n🔟 Price Bucket Distribution")
print(df10)



# ============================================
# ⏱️ SALES & MARKET PERFORMANCE
# ============================================

# 1️⃣ Average days on market by city
query11 = """
SELECT 
    l.City,
    AVG(s.Days_on_Market) AS Avg_Days
FROM sales_data s
JOIN listings l
ON s.Listing_ID = l.Listing_ID
GROUP BY l.City
"""

df11 = pd.read_sql(query11, conn)
print("\n1️⃣ Average Days on Market")
print(df11)



# 2️⃣ Fastest selling property types
query12 = """
SELECT 
    l.Property_Type,
    AVG(s.Days_on_Market) AS Avg_Days
FROM sales_data s
JOIN listings l
ON s.Listing_ID = l.Listing_ID
GROUP BY l.Property_Type
ORDER BY Avg_Days ASC
"""

df12 = pd.read_sql(query12, conn)
print("\n2️⃣ Fastest Selling Property Types")
print(df12)



# 3️⃣ Percentage sold above listing price
query13 = """
SELECT 
    ROUND(
        100 * SUM(
            CASE 
                WHEN s.Sale_Price > l.Price THEN 1
                ELSE 0
            END
        ) / COUNT(*),
        2
    ) AS Percentage_Above_Listing
FROM sales_data s
JOIN listings l
ON s.Listing_ID = l.Listing_ID
"""

df13 = pd.read_sql(query13, conn)
print("\n3️⃣ Percentage Sold Above Listing")
print(df13)



# 4️⃣ Sale-to-list ratio by city
query14 = """
SELECT 
    l.City,
    AVG(s.Sale_Price / l.Price) AS Sale_To_List_Ratio
FROM sales_data s
JOIN listings l
ON s.Listing_ID = l.Listing_ID
GROUP BY l.City
"""

df14 = pd.read_sql(query14, conn)
print("\n4️⃣ Sale-to-List Ratio")
print(df14)



# 5️⃣ Listings taking more than 90 days
query15 = """
SELECT 
    Listing_ID,
    Days_on_Market
FROM sales_data
WHERE Days_on_Market > 90
"""

df15 = pd.read_sql(query15, conn)
print("\n5️⃣ Listings > 90 Days")
print(df15)



# 6️⃣ Metro distance vs time on market
query16 = """
SELECT 
    p.metro_distance_km,
    AVG(s.Days_on_Market) AS Avg_Days
FROM property_attributes p
JOIN sales_data s
ON p.listing_id = s.Listing_ID
GROUP BY p.metro_distance_km
ORDER BY p.metro_distance_km
"""

df16 = pd.read_sql(query16, conn)
print("\n6️⃣ Metro Distance vs Market Time")
print(df16)



# 7️⃣ Monthly sales trend
query17 = """
SELECT 
    DATE_FORMAT(Date_Sold, '%Y-%m') AS Month,
    COUNT(*) AS Total_Sales
FROM sales_data
GROUP BY Month
ORDER BY Month
"""

df17 = pd.read_sql(query17, conn)
print("\n7️⃣ Monthly Sales Trend")
print(df17)



# 8️⃣ Unsold properties
query18 = """
SELECT 
    l.Listing_ID,
    l.City,
    l.Price
FROM listings l
LEFT JOIN sales_data s
ON l.Listing_ID = s.Listing_ID
WHERE s.Listing_ID IS NULL
"""

df18 = pd.read_sql(query18, conn)
print("\n8️⃣ Unsold Properties")
print(df18)



# ============================================
# 🧑‍💼 AGENT PERFORMANCE
# ============================================

# 1️⃣ Agents with most sales
query19 = """
SELECT 
    a.Name,
    COUNT(s.Listing_ID) AS Total_Sales
FROM agents a
JOIN listings l
ON a.Agent_ID = l.Agent_ID
JOIN sales_data s
ON l.Listing_ID = s.Listing_ID
GROUP BY a.Name
ORDER BY Total_Sales DESC
"""

df19 = pd.read_sql(query19, conn)
print("\n1️⃣ Agents with Most Sales")
print(df19)



# 2️⃣ Top agents by sales revenue
query20 = """
SELECT 
    a.Name,
    SUM(s.Sale_Price) AS Total_Revenue
FROM agents a
JOIN listings l
ON a.Agent_ID = l.Agent_ID
JOIN sales_data s
ON l.Listing_ID = s.Listing_ID
GROUP BY a.Name
ORDER BY Total_Revenue DESC
"""

df20 = pd.read_sql(query20, conn)
print("\n2️⃣ Top Agents by Revenue")
print(df20)

C:\Users\GLOBAL\AppData\Local\Temp\ipykernel_18780\1314273794.py:17: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df1 = pd.read_sql(query1, conn)
C:\Users\GLOBAL\AppData\Local\Temp\ipykernel_18780\1314273794.py:32: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df2 = pd.read_sql(query2, conn)
C:\Users\GLOBAL\AppData\Local\Temp\ipykernel_18780\1314273794.py:49: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df3 = pd.read_sql(query3, conn)
C:\Users\GLOBAL\AppData\Local\Temp\ipykernel_18780\1314273794.py:67: UserWarning: pandas


1️⃣ Average Listing Price by City
          City  Avg_Listing_Price
0  los angeles       1.093039e+06
1      houston       1.082539e+06
2      phoenix       1.071594e+06
3      chicago       1.065800e+06
4     new york       1.036485e+06

2️⃣ Average Price per Sqft by Property Type
  Property_Type  Avg_Price_Per_Sqft
0     apartment          523.303121
1     townhouse          580.424923
2         house          538.931991
3         condo          493.860388

3️⃣ Furnishing Status Impact
  furnishing_status     Avg_Price
0         furnished  1.069709e+06
1       unfurnished  1.066012e+06
2    semi-furnished  1.069909e+06


C:\Users\GLOBAL\AppData\Local\Temp\ipykernel_18780\1314273794.py:84: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df5 = pd.read_sql(query5, conn)



4️⃣ Metro Distance vs Price
     metro_distance_km     Avg_Price
0                 0.20  1.074458e+06
1                 0.21  1.076049e+06
2                 0.22  9.847907e+05
3                 0.23  1.081557e+06
4                 0.24  9.487864e+05
..                 ...           ...
776               7.96  1.067393e+06
777               7.97  1.215981e+06
778               7.98  1.158900e+06
779               7.99  1.058133e+06
780               8.00  1.054199e+06

[781 rows x 2 columns]


C:\Users\GLOBAL\AppData\Local\Temp\ipykernel_18780\1314273794.py:103: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df6 = pd.read_sql(query6, conn)
C:\Users\GLOBAL\AppData\Local\Temp\ipykernel_18780\1314273794.py:121: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df7 = pd.read_sql(query7, conn)
C:\Users\GLOBAL\AppData\Local\Temp\ipykernel_18780\1314273794.py:139: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df8 = pd.read_sql(query8, conn)
C:\Users\GLOBAL\AppData\Local\Temp\ipykernel_18780\1314273794.py:156: UserWarning: pa


5️⃣ Rented vs Non-Rented Prices
   is_rented     Avg_Price
0          1  1.069188e+06
1          0  1.067871e+06

6️⃣ Bedrooms & Bathrooms Effect
    bedrooms  bathrooms     Avg_Price
0          2          3  1.122915e+06
1          5          3  1.098165e+06
2          4          3  1.088822e+06
3          2          4  1.086213e+06
4          2          1  1.085803e+06
5          3          3  1.082559e+06
6          4          1  1.075772e+06
7          4          2  1.072362e+06
8          5          1  1.069201e+06
9          3          2  1.066504e+06
10         4          4  1.065664e+06
11         3          4  1.059472e+06
12         5          2  1.058238e+06
13         5          4  1.053989e+06
14         1          1  1.053920e+06
15         1          4  1.050851e+06
16         3          1  1.050392e+06
17         1          3  1.050164e+06
18         1          2  1.047136e+06
19         2          2  1.035652e+06

7️⃣ Parking & Power Backup Impact
   parking_available

C:\Users\GLOBAL\AppData\Local\Temp\ipykernel_18780\1314273794.py:179: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df10 = pd.read_sql(query10, conn)
C:\Users\GLOBAL\AppData\Local\Temp\ipykernel_18780\1314273794.py:200: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df11 = pd.read_sql(query11, conn)
C:\Users\GLOBAL\AppData\Local\Temp\ipykernel_18780\1314273794.py:218: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df12 = pd.read_sql(query12, conn)



2️⃣ Fastest Selling Property Types
  Property_Type   Avg_Days
0         house  58.337842
1     apartment  60.647670
2     townhouse  60.963653
3         condo  66.541176

3️⃣ Percentage Sold Above Listing
   Percentage_Above_Listing
0                     49.31

4️⃣ Sale-to-List Ratio
          City  Sale_To_List_Ratio
0     new york            0.999583
1      houston            1.000001
2      phoenix            0.998902
3  los angeles            0.999836
4      chicago            1.001527

5️⃣ Listings > 90 Days
    Listing_ID  Days_on_Market
0       l00004          115.99
1       l00007          108.01
2       l00024           91.02
3       l00028           91.99
4       l00031          107.99
..         ...             ...
184     l01151          109.01
185     l01159          101.98
186     l01166          106.01
187     l01178           98.99
188     l01195          118.01

[189 rows x 2 columns]

6️⃣ Metro Distance vs Market Time
     metro_distance_km   Avg_Days
0              

C:\Users\GLOBAL\AppData\Local\Temp\ipykernel_18780\1314273794.py:241: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df13 = pd.read_sql(query13, conn)
C:\Users\GLOBAL\AppData\Local\Temp\ipykernel_18780\1314273794.py:258: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df14 = pd.read_sql(query14, conn)
C:\Users\GLOBAL\AppData\Local\Temp\ipykernel_18780\1314273794.py:273: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df15 = pd.read_sql(query15, conn)
C:\Users\GLOBAL\AppData\Local\Temp\ipykernel_18780\1314273794.py:291: UserWarni